# Movie Recommendation System — Exploratory Notebook

This notebook demonstrates the full pipeline used to build a simple
**content-based movie recommendation system** using movie **genres**.

**Pipeline covered here:**
1. Load data (movies + ratings)
2. Basic Exploratory Data Analysis (EDA)
3. Preprocessing (clean genres, aggregate ratings)
4. TF-IDF vectorization of genres
5. Cosine similarity computation
6. Testing the recommendation function

> Note: The clean, reusable versions of this logic live in `src/data_loader.py`,
> `src/preprocess.py`, and `src/recommender.py`. This notebook is for
> exploration and verification only.


## 1. Setup — Imports and Path Configuration

In [ ]:
# Allow importing modules from the src/ folder
import sys
import os

sys.path.append(os.path.join(os.getcwd(), "..", "src"))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from data_loader import load_movies, load_ratings
from preprocess import clean_genres, aggregate_ratings, merge_movies_with_ratings, prepare_data

pd.set_option("display.max_colwidth", 100)


## 2. Load the Data

In [ ]:
movies_df = load_movies()
ratings_df = load_ratings()

print("Movies shape:", movies_df.shape)
print("Ratings shape:", ratings_df.shape)


In [ ]:
movies_df.head()

In [ ]:
ratings_df.head()

## 3. Basic Exploratory Data Analysis (EDA)

Before building any model, it's good practice to understand the data:
missing values, genre distribution, and rating distribution.


### 3.1 Missing Values Check

In [ ]:
print("Missing values in movies_df:")
print(movies_df.isnull().sum())

print("\nMissing values in ratings_df:")
print(ratings_df.isnull().sum())


### 3.2 How Many Movies Have No Genres Listed?

In [ ]:
no_genre_count = (movies_df["genres"] == "(no genres listed)").sum()
print(f"Movies with no genres listed: {no_genre_count} out of {len(movies_df)}")


### 3.3 Most Common Genres

In [ ]:
# Split genres on '|' and count occurrences of each individual genre
all_genres = movies_df["genres"].str.split("|").explode()
genre_counts = all_genres.value_counts()

genre_counts.head(15)


In [ ]:
genre_counts.head(15).plot(kind="barh", figsize=(8, 6))
plt.xlabel("Number of Movies")
plt.title("Most Common Genres")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()


### 3.4 Ratings Distribution

In [ ]:
ratings_df["rating"].plot(kind="hist", bins=10, figsize=(6, 4))
plt.xlabel("Rating")
plt.title("Distribution of Ratings")
plt.tight_layout()
plt.show()


In [ ]:
print("Average rating across all reviews:", round(ratings_df["rating"].mean(), 2))
print("Total number of ratings:", len(ratings_df))
print("Number of unique movies rated:", ratings_df["movieId"].nunique())


## 4. Preprocessing

Two steps:
1. Clean the `genres` column so it's usable by TF-IDF (replace `|` with spaces).
2. Aggregate ratings per movie (average rating + rating count) and merge
   them into the movies table — used only for **display**, not for similarity.


In [ ]:
movies_clean = clean_genres(movies_df)
movies_clean[["title", "genres", "genres_clean"]].head()


In [ ]:
rating_stats = aggregate_ratings(ratings_df)
rating_stats.head()


In [ ]:
movies_prepared = merge_movies_with_ratings(movies_clean, rating_stats)
movies_prepared[["title", "genres_clean", "avg_rating", "rating_count"]].head()


We can also do all three steps in a single call using `prepare_data()`:

In [ ]:
movies_prepared = prepare_data(movies_df, ratings_df)
movies_prepared.shape


## 5. TF-IDF Vectorization of Genres

We convert each movie's cleaned genre string into a numeric vector using
**TF-IDF (Term Frequency–Inverse Document Frequency)**. This treats each
genre as a "word" and each movie's genre list as a "document".

Movies that share more (and rarer) genres in common will end up with
more similar vectors.


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(token_pattern=r"[a-zA-Z\-]+")
tfidf_matrix = tfidf.fit_transform(movies_prepared["genres_clean"])

print("TF-IDF matrix shape:", tfidf_matrix.shape)
print("Number of unique genre terms:", len(tfidf.get_feature_names_out()))
print("Genre vocabulary:", list(tfidf.get_feature_names_out()))


## 6. Cosine Similarity Computation

We compute the cosine similarity between every pair of movies based on
their TF-IDF genre vectors. This produces an N x N matrix where cell
`[i, j]` represents how similar movie `i` is to movie `j`.


In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)
print("Cosine similarity matrix shape:", cosine_sim.shape)


## 7. Testing the Recommendation Function

Let's write a small function that, given a movie title, returns the
top-N most similar movies based on the cosine similarity matrix.

This mirrors the logic that will live in `src/recommender.py`.


In [ ]:
# Build a lookup from movie title -> DataFrame index
indices = pd.Series(movies_prepared.index, index=movies_prepared["title"]).drop_duplicates()


def get_recommendations(title, top_n=10):
    """Return top-N movies most similar in genre to the given title."""
    if title not in indices:
        return f"'{title}' not found in the dataset."

    idx = indices[title]

    # List of (index, similarity_score) pairs for this movie vs all others
    sim_scores = list(enumerate(cosine_sim[idx]))

    # Sort by similarity score, descending
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)

    # Skip the first result (it's the movie itself) and take the next top_n
    sim_scores = sim_scores[1: top_n + 1]

    movie_indices = [i[0] for i in sim_scores]

    return movies_prepared.loc[
        movie_indices, ["title", "genres_clean", "avg_rating", "rating_count"]
    ]


In [ ]:
# Test with a well-known movie from the dataset
get_recommendations("Toy Story (1995)", top_n=10)


In [ ]:
# Test with another movie to sanity-check different genres
get_recommendations("Jumanji (1995)", top_n=5)


In [ ]:
# Test graceful handling of a movie that doesn't exist
get_recommendations("This Movie Does Not Exist (2099)")


## 8. Saving the Similarity Matrix (Reference Only)

The actual pickling of the similarity matrix and movie data is handled
by `src/recommender.py` (Phase 4), so the Streamlit app can load it
without recomputing everything. This cell just demonstrates the idea.


In [ ]:
import pickle

# Example only — the real save logic lives in src/recommender.py
# with open("../models/cosine_similarity.pkl", "wb") as f:
#     pickle.dump(cosine_sim, f)

print("Pickling logic will be implemented in src/recommender.py (Phase 4).")


## 9. Summary

- Loaded `movies.csv` and `ratings.csv` from the MovieLens `ml-latest-small` dataset.
- Explored genre distribution and rating distribution.
- Cleaned genres and aggregated ratings for display purposes.
- Converted genres into TF-IDF vectors.
- Computed cosine similarity between all movie pairs.
- Verified the recommendation function works correctly, including
  graceful handling of an invalid movie title.

Next step: move this logic into `src/recommender.py` as reusable,
production-ready functions (Phase 4).
